# FraudGuard-ML — Exploratory Data Analysis

This notebook explores the [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset (ULB) and summarizes key findings that guide model training in `train.py`.

**Scope:** EDA, visualizations, and a high-level model comparison overview.  
Full training, threshold tuning, and model persistence live in `train.py` — this notebook does not reimplement the entire pipeline.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)

DATA_PATH = Path("..") / "data" / "creditcard.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}.\n"
        "Download creditcard.csv from Kaggle and place it in the data/ folder.\n"
        "See data/README.md for details."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH} successfully.")

## 1. Dataset Overview

We inspect shape, sample rows, dtypes, missing values, duplicates, and basic statistics.

In [ ]:
print("Shape:", df.shape)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
duplicates = df.duplicated().sum()

print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values.")
print(f"\nDuplicate rows: {duplicates:,}")

In [ ]:
df.describe().T

**Findings**

- Features `V1`–`V28` are PCA-transformed anonymized variables; `Time` and `Amount` remain in their original scale.
- The dataset is typically complete (no missing values) but may contain a small number of duplicate rows.
- `Amount` is heavily right-skewed — most transactions are small.

## 2. Class Distribution

Fraud is rare. Accuracy alone is misleading on this problem; precision, recall, F1, ROC-AUC, and Average Precision (AP) are more informative.

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
n_legit = int(class_counts.get(0, 0))
n_fraud = int(class_counts.get(1, 0))
fraud_pct = n_fraud / len(df) * 100

print(f"Legitimate transactions : {n_legit:,}")
print(f"Fraudulent transactions : {n_fraud:,}")
print(f"Fraud percentage        : {fraud_pct:.4f}%")

fig, ax = plt.subplots()
bars = ax.bar(
    ["Legitimate (0)", "Fraud (1)"],
    [n_legit, n_fraud],
    color=["#2E7D32", "#C62828"],
)
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
for bar, count in zip(bars, [n_legit, n_fraud]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()

## 3. Transaction Amount Analysis

Compare amount distributions for legitimate vs fraudulent transactions.

In [ ]:
legit = df[df["Class"] == 0]["Amount"]
fraud = df[df["Class"] == 1]["Amount"]

print("Amount — Legitimate")
print(legit.describe())
print("\nAmount — Fraud")
print(fraud.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot (log-scaled y for readability)
sns.boxplot(
    data=df,
    x="Class",
    y="Amount",
    hue="Class",
    palette={0: "#2E7D32", 1: "#C62828"},
    legend=False,
    ax=axes[0],
)
axes[0].set_yscale("log")
axes[0].set_xticklabels(["Legitimate", "Fraud"])
axes[0].set_title("Amount by Class (log scale)")
axes[0].set_xlabel("")

# Histogram of amounts (capped for readability)
cap = df["Amount"].quantile(0.99)
sns.histplot(
    df[df["Amount"] <= cap],
    x="Amount",
    hue="Class",
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
    palette={0: "#2E7D32", 1: "#C62828"},
    ax=axes[1],
)
axes[1].set_title(f"Amount Density (≤ 99th percentile ≈ {cap:.0f})")

plt.tight_layout()
plt.show()

**Findings**

- Fraudulent amounts are not always large; many frauds occur at modest values.
- Amount alone is a weak discriminator — PCA features carry most of the signal.

## 4. Transaction Time Analysis

`Time` is **elapsed seconds since the first transaction** in the dataset — not a clock timestamp. We explore activity patterns over this relative timeline.

In [ ]:
df["Hours"] = df["Time"] / 3600

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    df[df["Class"] == 0]["Hours"],
    bins=48,
    color="#2E7D32",
    ax=axes[0],
)
axes[0].set_title("Legitimate Transactions over Time")
axes[0].set_xlabel("Hours since first transaction")

sns.histplot(
    df[df["Class"] == 1]["Hours"],
    bins=48,
    color="#C62828",
    ax=axes[1],
)
axes[1].set_title("Fraudulent Transactions over Time")
axes[1].set_xlabel("Hours since first transaction")

plt.tight_layout()
plt.show()

print(f"Time span: {df['Time'].min():.0f}s → {df['Time'].max():.0f}s "
      f"({df['Hours'].max():.1f} hours)")

**Findings**

- Legitimate volume shows clear periodic structure across the ~48-hour window.
- Fraud occurs throughout the period and does not cluster into a single obvious time window.

## 5. Correlation Analysis

Because `V1`–`V28` come from PCA, they should be largely uncorrelated with each other. Correlations with `Class` help highlight which components relate most to fraud.

In [ ]:
corr = df.drop(columns=["Hours"], errors="ignore").corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
target_corr = (
    corr["Class"]
    .drop("Class")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Top features correlated with Class (by absolute value):")
print(target_corr.head(10).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
top = target_corr.head(10)
colors = ["#C62828" if v < 0 else "#1565C0" for v in top.values]
top.plot(kind="barh", color=colors, ax=ax)
ax.set_xlabel("Correlation with Class")
ax.set_title("Top 10 Feature–Target Correlations")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Modeling Notes (see `train.py`)

Training is implemented in `train.py` with this design:

| Step | Approach |
|---|---|
| Split | Stratified 70% / 15% / 15% (train / val / test), `random_state=42` |
| Imbalance | `class_weight="balanced"` (LogReg) and `scale_pos_weight` from **train** counts (XGBoost) |
| Scaling | `StandardScaler` inside a Pipeline for Logistic Regression only |
| Selection | Best model by **validation Average Precision** |
| Threshold | Maximize **validation F1**; test set used once for final reporting |

Run training from the project root:

```bash
python train.py
```

Metrics and figures are written to `models/`.

## 7. Summary of Key Findings

1. **Severe class imbalance** (~0.17% fraud) → prioritize AP / PR metrics over accuracy.
2. **No missing values** in the standard dataset; duplicates may exist but are minor.
3. **Amount** is skewed and only weakly discriminative alone.
4. **Time** is relative elapsed seconds — useful as a feature, not a clock.
5. **PCA features** dominate the correlation signal with fraud labels.
6. Class-weighted learning is a simple, effective starting point before resampling methods.